In [ ]:
import pandas as pd
from pathlib import Path
import re

# ============================================================
# SOURCE FOLDER
# ============================================================

input_folder = Path(
    r"D:\Gait Dataset\200GaitData — 100 people\Output-20260824T102931Z-1-001\Output"
)

# Folder where combined datasets will be saved
output_folder = input_folder / "Combined_RightPocket"
output_folder.mkdir(exist_ok=True)

# Output files
session1_output = output_folder / "right_pocket_session1_all_subjects.csv"
session2_output = output_folder / "right_pocket_session2_all_subjects.csv"


# ============================================================
# FUNCTION TO READ AND PROCESS ONE FILE
# ============================================================

def process_file(file_path, subject_id):
    try:
        df = pd.read_csv(file_path)

        # Remove automatically generated CSV index columns
        unnamed_columns = [
            col for col in df.columns
            if str(col).startswith("Unnamed:")
        ]

        if unnamed_columns:
            df = df.drop(columns=unnamed_columns)

        # Add subject ID
        df["Subject"] = subject_id

        return df

    except Exception as e:
        print(f"ERROR reading {file_path.name}: {e}")
        return None


# ============================================================
# FIND RIGHT POCKET FILES
# ============================================================

session1_data = []
session2_data = []

session1_subjects = []
session2_subjects = []

all_files = list(input_folder.glob("*.csv"))

print(f"\nTotal CSV files found: {len(all_files)}\n")
print("=" * 70)


for file_path in all_files:

    filename = file_path.stem.lower()

    # Match:
    # sub1-rp-s1
    # sub23-rp-s2
    # sub93-rp-s1
    match = re.fullmatch(r"sub(\d+)-rp-s([12])", filename)

    if not match:
        continue

    subject_id = int(match.group(1))
    session = int(match.group(2))

    print(
        f"Processing: {file_path.name:25s} "
        f"Subject = {subject_id:3d} | Session = {session}"
    )

    df = process_file(file_path, subject_id)

    if df is None:
        continue

    if session == 1:
        session1_data.append(df)
        session1_subjects.append(subject_id)

    elif session == 2:
        session2_data.append(df)
        session2_subjects.append(subject_id)


# ============================================================
# COMBINE SESSION 1
# ============================================================

if session1_data:

    combined_session1 = pd.concat(
        session1_data,
        ignore_index=True
    )

    combined_session1.to_csv(
        session1_output,
        index=False
    )

    print("\n" + "=" * 70)
    print("SESSION 1 CREATED")
    print("=" * 70)

    print(f"Subjects      : {len(set(session1_subjects))}")
    print(f"Total rows    : {len(combined_session1):,}")
    print(f"Columns       : {list(combined_session1.columns)}")
    print(f"Saved to      : {session1_output}")

else:
    print("\nNo Right Pocket Session 1 files found.")


# ============================================================
# COMBINE SESSION 2
# ============================================================

if session2_data:

    combined_session2 = pd.concat(
        session2_data,
        ignore_index=True
    )

    combined_session2.to_csv(
        session2_output,
        index=False
    )

    print("\n" + "=" * 70)
    print("SESSION 2 CREATED")
    print("=" * 70)

    print(f"Subjects      : {len(set(session2_subjects))}")
    print(f"Total rows    : {len(combined_session2):,}")
    print(f"Columns       : {list(combined_session2.columns)}")
    print(f"Saved to      : {session2_output}")

else:
    print("\nNo Right Pocket Session 2 files found.")


# ============================================================
# CHECK MISSING SUBJECTS
# ============================================================

expected_subjects = set(range(1, 101))

found_s1 = set(session1_subjects)
found_s2 = set(session2_subjects)

missing_s1 = sorted(expected_subjects - found_s1)
missing_s2 = sorted(expected_subjects - found_s2)

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(f"Session 1 subjects found : {len(found_s1)}")
print(f"Session 2 subjects found : {len(found_s2)}")

print(f"\nSession 1 missing subjects: {missing_s1}")
print(f"Session 2 missing subjects: {missing_s2}")

print("\nDone!")